# Getting Started with segment-geospatial (samgeo)

[segment-geospatial](https://github.com/opengeos/segment-geospatial) wraps Meta's Segment Anything Model (SAM) so it works natively with georeferenced raster data (GeoTIFFs). It handles coordinate reference systems, tiling, and mask export automatically.

**When to use samgeo:** You need to delineate objects (buildings, fields, water bodies, tree crowns) in satellite or aerial imagery but have **no labeled training data**. SAM is a zero-shot model — you supply a prompt (point, box, or text) and it segments the object.

## References
- Documentation: https://samgeo.gishub.org/
- GitHub: https://github.com/opengeos/segment-geospatial

## 1. Verify Installation

In [ ]:
import samgeo
print(samgeo.__version__)

## 2. Load a SAM Model

Available checkpoints: `vit_h` (~2.4 GB), `vit_l` (~1.2 GB), `vit_b` (~370 MB). Weights are downloaded automatically on first use and cached in `~/.cache/torch/hub/`.

In [ ]:
from samgeo import SamGeo

# Use vit_b for faster experimentation; switch to vit_h for best accuracy
sam = SamGeo(
    model_type="vit_b",
    automatic=False,  # We will supply prompts manually in this notebook
)

## 3. Download Sample Imagery

We use a small aerial RGB GeoTIFF from the samgeo documentation examples.

In [ ]:
import os
import urllib.request

url = "https://github.com/opengeos/samgeo/releases/download/v0.2.0/aerial_image.tif"
image_path = "aerial_image.tif"

if not os.path.exists(image_path):
    print("Downloading sample image...")
    urllib.request.urlretrieve(url, image_path)
    print("Done.")
else:
    print("Image already cached.")

## 4. Visualize the Image

In [ ]:
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import show

with rasterio.open(image_path) as src:
    fig, ax = plt.subplots(figsize=(8, 8))
    show(src, ax=ax)
    ax.set_title("Sample Aerial Image")
    plt.tight_layout()
    plt.show()
    print(f"CRS: {src.crs}")
    print(f"Shape: {src.height} x {src.width} px")
    print(f"Bands: {src.count}")

## 5. Segment with Point Prompts

Supply one or more (x, y) pixel coordinate pairs as foreground prompts. SAM delineates the object at each point.

In [ ]:
sam.set_image(image_path)

# Foreground points: (column, row) pixel coordinates
point_coords = [[200, 200], [400, 300]]
point_labels = [1, 1]  # 1 = foreground, 0 = background

masks, scores, logits = sam.predict(
    point_coords=point_coords,
    point_labels=point_labels,
)

print(f"Generated {len(masks)} mask candidates")
print(f"Confidence scores: {scores}")

In [ ]:
import numpy as np

fig, axes = plt.subplots(1, len(masks), figsize=(5 * len(masks), 5))
if len(masks) == 1:
    axes = [axes]

for i, (mask, score) in enumerate(zip(masks, scores)):
    axes[i].imshow(mask, cmap="gray")
    axes[i].set_title(f"Mask {i+1}  (score={score:.3f})")
    axes[i].axis("off")

plt.suptitle("SAM mask candidates (best = highest score)")
plt.tight_layout()
plt.show()

## 6. Segment with a Bounding Box Prompt

A bounding box prompt `[x_min, y_min, x_max, y_max]` in pixel coordinates gives SAM a tighter spatial prior than a point.

In [ ]:
box = [150, 150, 450, 400]  # [x_min, y_min, x_max, y_max]

masks_box, scores_box, _ = sam.predict(box=box)

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(masks_box[0], cmap="gray")
ax.set_title(f"Box-prompted mask  (score={scores_box[0]:.3f})")
ax.axis("off")
plt.tight_layout()
plt.show()

## 7. Export Mask as GeoTIFF

samgeo preserves the CRS and geotransform of the source image when saving masks.

In [ ]:
sam.save_mask(masks_box[0], "output_mask.tif", image=image_path)

with rasterio.open("output_mask.tif") as src:
    print(f"Output CRS: {src.crs}")
    print(f"Output transform: {src.transform}")
    print("Mask saved successfully with geospatial metadata.")

## Next Steps

- `1-automatic_mask_generation.ipynb` — generate masks for the entire image without any prompts
- `2-text_prompted_segmentation.ipynb` — use natural language prompts ("building", "tree") via LangSAM
- `3-batch_inference_geotiff.ipynb` — tiled inference on large imagery